# Testing NBodySimulation
This notebook initializes and runs the `NBodySimulation` model, solving for positions, velocities, and conserving energy.

In [ ]:
import os
if(os.getcwd().split('/')[-1] == 'notebooks'):
  os.chdir('..')

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from simulation import NBodySimulation, FluidSimulation

%matplotlib inline

In [ ]:
# Initialize a 3-body system
sim = FluidSimulation(
    num_particles=2000,
    gravitational_constant=9.81,
    softening_length=0.015,
    stiffness=1000,
    exponent=7.0,
    viscosity=0.3
)

position_scale=0.5

np.random.seed(42)
sim.initialize_random_state(
    position_scale=position_scale,
    velocity_scale=0.0,
    mass_range=(1.0, 1.0),
    start=(1,1)
)

init_energy = sim.compute_total_energy()
print(f"Initial Total Energy: {init_energy:.5f}")

In [ ]:
# Simulate
dt = 0.0005
total_time = 5.0
fps = 60
save_every = round((total_time/dt)/(total_time*fps))

history_positions, history_velocities, history_times = sim.simulate(
    total_time=total_time,
    dt=dt,
    save_every=save_every
)

final_energy = sim.compute_total_energy()
print(f"Final Total Energy: {final_energy:.5f}")
print(f"Relative Energy Error: {abs(final_energy - init_energy)/abs(init_energy):.2e}")

In [ ]:
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib import colormaps
from matplotlib import colors
from IPython.display import HTML, Video

# Set up the animation figure
fig, ax = plt.subplots(figsize=(8, 8))

# Pre-calculate plot limits to keep the camera still
all_pos = history_positions

step_size = 1
all_pos = all_pos[::step_size]

ax.set_xlim(0 - 0.05, position_scale + 0.05)
ax.set_ylim(0 - 0.05, position_scale + 0.05)
ax.set_xlabel('X Position')
ax.set_ylabel('Y Position')
title_str = f'Simulation ({sim.num_particles} particles)'
ax.set_title(title_str)

# Calculate dynamic markersize in points to match physical particle size
particle_diameter = getattr(sim, 'smoothing_length', getattr(sim, 'softening_length', 0.07))
p1 = ax.transData.transform((0, 0))
p2 = ax.transData.transform((particle_diameter, 0))
marker_size_pt = (p2[0] - p1[0]) * 72.0 / fig.dpi /2
ax.grid(False)

# Create a gradient from blue to deep blue (ocean-like)
water_colormap = colors.LinearSegmentedColormap.from_list('ocean_blue', ['#000033', '#0088cc'])

colors = np.arange(sim.num_particles) # Gradient by particle index to track motion
points = ax.scatter(all_pos[0, :, 0], all_pos[0, :, 1], c=colors, cmap=water_colormap, s=marker_size_pt**2)

def init():
    points.set_offsets(all_pos[0, :, :2])
    return points,

def update(frame):
    points.set_offsets(all_pos[frame, :, :2])
    return points,

num_frames = all_pos.shape[0]
ani = FuncAnimation(fig, update, frames=num_frames, init_func=init, blit=True)
plt.close(fig) # Prevent static plot from showing

# Save animation efficiently directly to mp4 using the FFMpegWriter
writer = FFMpegWriter(fps=fps, codec='h264', bitrate=1200, extra_args=['-preset', 'ultrafast', '-tune', 'fastdecode'])
ani.save('simulation.mp4', writer=writer)

# Display the local file using IPython.display.Video which is much faster than to_html5_video base64 encoding
Video('simulation.mp4', embed=True, html_attributes='controls autoplay loop')